# Epic 1.3: MA Crossover - Dataset Creation

**Alternative Approach to HMM Regime Detection**

This notebook creates the MAML dataset using **Moving Average Crossover** instead of HMM for regime detection.

## Approach:
- **Regime Detection**: MA-20 vs MA-50 crossover
- **Regime 0 (Bearish)**: MA-20 < MA-50 (downtrend)
- **Regime 1 (Bullish)**: MA-20 > MA-50 (uptrend)

## What This Notebook Does:
1. ✅ Calculate MA-20 and MA-50 from SP500 close prices
2. ✅ Detect regime switches (crossovers)
3. ✅ Segment consecutive days by regime
4. ✅ Filter segments ≥40 days (30 support + 10 query)
5. ✅ Create PyTorch Dataset for MAML
6. ✅ Save outputs: `regime_labels_ma.csv`, `regime_segments_ma.pkl`

**Ready to run!** 🚀

## Step 1: Import Libraries & Load Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import torch
from torch.utils.data import Dataset
from datetime import datetime

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [ ]:
# Load cleaned SP500/VIX data
df = pd.read_csv('../data/processed/sp500_vix_merged_clean.csv', index_col='Date', parse_dates=True)

# Load features for MAML tasks
features_df = pd.read_csv('../data/processed/features_imputed.csv', index_col='date', parse_dates=True)

print(f"Loaded SP500/VIX data: {df.shape}")
print(f"Loaded features data: {features_df.shape}")
print(f"\nDate range: {df.index.min()} to {df.index.max()}")
print(f"Total days: {len(df)}")

df.head()

## Step 2: Calculate Moving Averages

Calculate MA-20 (fast) and MA-50 (slow) from SP500 close prices.

In [ ]:
# Extract close price (find the correct column)
close_col = [col for col in df.columns if 'close' in col.lower() and 'sp500' in col.lower()][0]
close = df[close_col]

print(f"Using column: {close_col}")
print(f"Close price range: {close.min():.2f} to {close.max():.2f}")

# Calculate moving averages
df['MA_20'] = close.rolling(window=20, min_periods=20).mean()
df['MA_50'] = close.rolling(window=50, min_periods=50).mean()

# Remove NaN rows (first 50 days don't have MA_50)
df_clean = df.dropna(subset=['MA_20', 'MA_50'])

print(f"\nAfter removing NaN (first 50 days):")
print(f"Clean data: {len(df_clean)} days")
print(f"Date range: {df_clean.index.min()} to {df_clean.index.max()}")

df_clean[['MA_20', 'MA_50']].head(10)

## Step 3: Detect Regimes (MA Crossover)

- **Regime 0 (Bearish)**: MA-20 < MA-50 (death cross)
- **Regime 1 (Bullish)**: MA-20 > MA-50 (golden cross)

In [ ]:
# Detect regime: 1 if MA_20 > MA_50 (bullish), 0 otherwise (bearish)
df_clean['regime'] = (df_clean['MA_20'] > df_clean['MA_50']).astype(int)

# Detect regime switches
df_clean['regime_switch'] = df_clean['regime'].diff().abs()

# Count regime switches
num_switches = int(df_clean['regime_switch'].sum())
regime_0_days = (df_clean['regime'] == 0).sum()
regime_1_days = (df_clean['regime'] == 1).sum()

print(f"MA Crossover Regime Detection Results:")
print(f"=" * 50)
print(f"Total regime switches: {num_switches}")
print(f"Regime 0 (Bearish) days: {regime_0_days} ({regime_0_days/len(df_clean)*100:.1f}%)")
print(f"Regime 1 (Bullish) days: {regime_1_days} ({regime_1_days/len(df_clean)*100:.1f}%)")
print(f"\nAverage regime duration: {len(df_clean) / (num_switches + 1):.1f} days")

df_clean[['regime', 'regime_switch']].head(20)

## Step 4: Visualize MA Crossover Regimes

Plot SP500 price with MA overlays and regime shading.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# Plot 1: Price + Moving Averages + Regime Shading
axes[0].plot(df_clean.index, close.loc[df_clean.index], label='SP500 Close', 
             alpha=0.6, linewidth=1.5, color='black')
axes[0].plot(df_clean.index, df_clean['MA_20'], label='MA-20 (Fast)', 
             color='orange', linewidth=2)
axes[0].plot(df_clean.index, df_clean['MA_50'], label='MA-50 (Slow)', 
             color='red', linewidth=2)

# Shade regimes
bullish_mask = df_clean['regime'] == 1
bearish_mask = df_clean['regime'] == 0
axes[0].fill_between(df_clean.index, close.loc[df_clean.index].min(), close.loc[df_clean.index].max(),
                      where=bullish_mask, alpha=0.2, color='green', label='Bullish Regime')
axes[0].fill_between(df_clean.index, close.loc[df_clean.index].min(), close.loc[df_clean.index].max(),
                      where=bearish_mask, alpha=0.2, color='red', label='Bearish Regime')

axes[0].set_title('SP500 Price with MA Crossover Regimes', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price ($)')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# Plot 2: MA Distance (MA_20 - MA_50)
ma_distance = df_clean['MA_20'] - df_clean['MA_50']
axes[1].plot(df_clean.index, ma_distance, color='purple', linewidth=1.5)
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].fill_between(df_clean.index, 0, ma_distance, where=(ma_distance > 0),
                      alpha=0.3, color='green', label='MA-20 > MA-50 (Bullish)')
axes[1].fill_between(df_clean.index, 0, ma_distance, where=(ma_distance < 0),
                      alpha=0.3, color='red', label='MA-20 < MA-50 (Bearish)')
axes[1].set_title('MA Crossover Distance (MA-20 minus MA-50)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Distance ($)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Regime Timeline
axes[2].fill_between(df_clean.index, 0, 1, where=bullish_mask, 
                      alpha=0.7, color='green', label='Bullish')
axes[2].fill_between(df_clean.index, 0, 1, where=bearish_mask, 
                      alpha=0.7, color='red', label='Bearish')
axes[2].set_title('Regime Timeline', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Regime')
axes[2].set_yticks([0, 1])
axes[2].set_yticklabels(['Bearish', 'Bullish'])
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/figures/ma_crossover_regimes.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Visualization saved to docs/figures/ma_crossover_regimes.png")

## Step 5: Segment Consecutive Days by Regime

Group consecutive days with the same regime into segments.

In [ ]:
# Segment consecutive days by regime
segments_raw = []
current_regime = df_clean['regime'].iloc[0]
start_idx = 0

for i in range(1, len(df_clean)):
    if df_clean['regime'].iloc[i] != current_regime:
        # Save completed segment
        segments_raw.append({
            'start_idx': start_idx,
            'end_idx': i - 1,
            'regime_label': int(current_regime),
            'length': i - start_idx,
            'start_date': df_clean.index[start_idx],
            'end_date': df_clean.index[i - 1]
        })
        # Start new segment
        start_idx = i
        current_regime = df_clean['regime'].iloc[i]

# Don't forget the last segment
segments_raw.append({
    'start_idx': start_idx,
    'end_idx': len(df_clean) - 1,
    'regime_label': int(current_regime),
    'length': len(df_clean) - start_idx,
    'start_date': df_clean.index[start_idx],
    'end_date': df_clean.index[-1]
})

print(f"Total segments (before filtering): {len(segments_raw)}")
print(f"\nFirst 10 segments:")
for i, seg in enumerate(segments_raw[:10]):
    regime_name = "Bullish" if seg['regime_label'] == 1 else "Bearish"
    print(f"  {i+1}. {regime_name}: {seg['length']} days ({seg['start_date'].date()} to {seg['end_date'].date()})")

## Step 6: Filter Segments (≥40 days)

Keep only segments with at least 40 days (30 for support + 10 for query).

In [ ]:
# Filter: keep only segments with length >= 40 days
MIN_SEGMENT_LENGTH = 40
segments = [seg for seg in segments_raw if seg['length'] >= MIN_SEGMENT_LENGTH]

# Statistics
regime_0_segs = [s for s in segments if s['regime_label'] == 0]
regime_1_segs = [s for s in segments if s['regime_label'] == 1]
avg_length = np.mean([s['length'] for s in segments])
median_length = np.median([s['length'] for s in segments])

print(f"Segment Filtering Results:")
print(f"=" * 50)
print(f"Segments before filtering: {len(segments_raw)}")
print(f"Segments after ≥{MIN_SEGMENT_LENGTH} day filter: {len(segments)}")
print(f"\nRegime 0 (Bearish) segments: {len(regime_0_segs)}")
print(f"Regime 1 (Bullish) segments: {len(regime_1_segs)}")
print(f"\nAverage segment length: {avg_length:.1f} days")
print(f"Median segment length: {median_length:.1f} days")
print(f"Shortest segment: {min([s['length'] for s in segments])} days")
print(f"Longest segment: {max([s['length'] for s in segments])} days")

# Display all filtered segments
print(f"\n✅ All {len(segments)} filtered segments:")
print(f"{'#':<4} {'Regime':<10} {'Length':<8} {'Start Date':<12} {'End Date':<12}")
print("-" * 50)
for i, seg in enumerate(segments):
    regime_name = "Bullish" if seg['regime_label'] == 1 else "Bearish"
    print(f"{i+1:<4} {regime_name:<10} {seg['length']:<8} {seg['start_date'].date()} {seg['end_date'].date()}")

## Step 7: Visualize Segment Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Histogram of segment lengths
lengths = [s['length'] for s in segments]
axes[0].hist(lengths, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(MIN_SEGMENT_LENGTH, color='red', linestyle='--', linewidth=2, 
                label=f'Min threshold ({MIN_SEGMENT_LENGTH} days)')
axes[0].axvline(avg_length, color='green', linestyle='--', linewidth=2, 
                label=f'Mean ({avg_length:.1f} days)')
axes[0].set_xlabel('Segment Length (days)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Distribution of Segment Lengths', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Bar chart of segments per regime
regime_counts = [len(regime_0_segs), len(regime_1_segs)]
regime_labels = ['Regime 0\n(Bearish)', 'Regime 1\n(Bullish)']
colors = ['#e74c3c', '#2ecc71']
bars = axes[1].bar(regime_labels, regime_counts, color=colors, alpha=0.7, edgecolor='black', linewidth=2)

# Add count labels on bars
for bar, count in zip(bars, regime_counts):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                 f'{count}', ha='center', va='bottom', fontsize=14, fontweight='bold')

axes[1].set_ylabel('Number of Segments', fontsize=12)
axes[1].set_title('Segments by Regime', fontsize=14, fontweight='bold')
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/figures/ma_segment_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Segment distribution saved to docs/figures/ma_segment_distribution.png")

## Step 8: Save Regime Labels & Segments

In [ ]:
# Save regime labels (CSV format - same as HMM output)
regime_output = df_clean[['regime', 'MA_20', 'MA_50']].copy()
regime_output.columns = ['regime', 'ma_20', 'ma_50']
regime_output.to_csv('../data/processed/regime_labels_ma.csv')

print(f"✅ Saved regime labels to: data/processed/regime_labels_ma.csv")
print(f"   Shape: {regime_output.shape}")

# Save segments (pickle format - same as HMM output)
with open('../data/processed/regime_segments_ma.pkl', 'wb') as f:
    pickle.dump(segments, f)

print(f"✅ Saved {len(segments)} segments to: data/processed/regime_segments_ma.pkl")

# Display sample of saved data
print(f"\nSample of regime_labels_ma.csv:")
print(regime_output.head(10))

## Step 9: Create MAML Task Dataset

Create PyTorch Dataset with sliding windows for meta-learning.

In [ ]:
class RegimeTaskDataset(Dataset):
    """
    PyTorch Dataset for MAML meta-learning tasks from MA Crossover regimes.
    
    Each task contains:
    - Support set: 30 consecutive days (for adaptation)
    - Query set: Next 10 consecutive days (for evaluation)
    - All within a single regime segment
    """
    
    def __init__(self, segments, features_df, support_size=30, query_size=10, stride=20):
        """
        Args:
            segments: List of segment dicts from MA crossover detection
            features_df: DataFrame with features (11 cols) and targets (11 cols)
            support_size: Number of days in support set (default: 30)
            query_size: Number of days in query set (default: 10)
            stride: Sliding window stride in days (default: 20 for 50% overlap)
        """
        self.segments = segments
        self.features_df = features_df
        self.support_size = support_size
        self.query_size = query_size
        self.stride = stride
        self.window_size = support_size + query_size
        
        # Identify feature and target columns
        self.feature_cols = [col for col in features_df.columns if not col.endswith('_target')]
        self.target_cols = [col for col in features_df.columns if col.endswith('_target')]
        
        # Generate all tasks
        self.tasks = self._generate_tasks()
        
    def _generate_tasks(self):
        """Generate all possible tasks from segments using sliding windows."""
        tasks = []
        
        for seg_id, seg in enumerate(self.segments):
            seg_length = seg['length']
            
            # Slide window through segment
            for start_offset in range(0, seg_length - self.window_size + 1, self.stride):
                start_idx = seg['start_idx'] + start_offset
                
                tasks.append({
                    'start_idx': start_idx,
                    'segment_id': seg_id,
                    'regime_label': seg['regime_label']
                })
        
        return tasks
    
    def __len__(self):
        return len(self.tasks)
    
    def __getitem__(self, idx):
        """
        Get a single task.
        
        Returns:
            dict with keys:
                - support_x: torch.Tensor (30, 11) - features
                - support_y: torch.Tensor (30, 11) - targets
                - query_x: torch.Tensor (10, 11) - features
                - query_y: torch.Tensor (10, 11) - targets
                - regime_label: int (0=bearish or 1=bullish)
                - segment_id: int
        """
        task = self.tasks[idx]
        start_idx = task['start_idx']
        
        # Extract support set (30 days)
        support_data = self.features_df.iloc[start_idx:start_idx + self.support_size]
        support_x = torch.tensor(support_data[self.feature_cols].values, dtype=torch.float32)
        support_y = torch.tensor(support_data[self.target_cols].values, dtype=torch.float32)
        
        # Extract query set (10 days)
        query_data = self.features_df.iloc[start_idx + self.support_size:start_idx + self.window_size]
        query_x = torch.tensor(query_data[self.feature_cols].values, dtype=torch.float32)
        query_y = torch.tensor(query_data[self.target_cols].values, dtype=torch.float32)
        
        return {
            'support_x': support_x,
            'support_y': support_y,
            'query_x': query_x,
            'query_y': query_y,
            'regime_label': task['regime_label'],
            'segment_id': task['segment_id']
        }

print("✅ RegimeTaskDataset class defined!")

## Step 10: Create Dataset Instance & Test

In [ ]:
# Create dataset
dataset = RegimeTaskDataset(segments, features_df, support_size=30, query_size=10, stride=20)

print(f"MA Crossover MAML Dataset Created!")
print(f"=" * 50)
print(f"Total tasks: {len(dataset)}")
print(f"Total segments: {len(segments)}")
print(f"Average tasks per segment: {len(dataset) / len(segments):.2f}")

# Test first task
task = dataset[0]
print(f"\n✅ First task structure:")
print(f"  Support X shape: {task['support_x'].shape}")
print(f"  Support Y shape: {task['support_y'].shape}")
print(f"  Query X shape: {task['query_x'].shape}")
print(f"  Query Y shape: {task['query_y'].shape}")
print(f"  Regime: {task['regime_label']} ({'Bullish' if task['regime_label'] == 1 else 'Bearish'})")
print(f"  Segment ID: {task['segment_id']}")

# Check for NaN values
has_nan = (torch.isnan(task['support_x']).any() or 
           torch.isnan(task['support_y']).any() or
           torch.isnan(task['query_x']).any() or
           torch.isnan(task['query_y']).any())

print(f"\n  Contains NaN: {'❌ Yes' if has_nan else '✅ No'}")

## Step 11: Validate Dataset Quality

In [ ]:
# Comprehensive validation
print("Running comprehensive dataset validation...")
print("=" * 50)

# Check all tasks for NaNs
nan_count = 0
for i in range(len(dataset)):
    task = dataset[i]
    if (torch.isnan(task['support_x']).any() or torch.isnan(task['query_x']).any()):
        nan_count += 1

print(f"✅ NaN Check: {nan_count} / {len(dataset)} tasks contain NaN")

# Check regime distribution
regime_labels = [dataset[i]['regime_label'] for i in range(len(dataset))]
regime_0_tasks = sum(1 for r in regime_labels if r == 0)
regime_1_tasks = sum(1 for r in regime_labels if r == 1)

print(f"\n✅ Regime Distribution:")
print(f"   Regime 0 (Bearish): {regime_0_tasks} tasks ({regime_0_tasks/len(dataset)*100:.1f}%)")
print(f"   Regime 1 (Bullish): {regime_1_tasks} tasks ({regime_1_tasks/len(dataset)*100:.1f}%)")

# Check feature value ranges
all_support_x = torch.stack([dataset[i]['support_x'] for i in range(min(100, len(dataset)))])
all_query_x = torch.stack([dataset[i]['query_x'] for i in range(min(100, len(dataset)))])

print(f"\n✅ Feature Value Ranges (first 100 tasks):")
print(f"   Support X - Min: {all_support_x.min():.3f}, Max: {all_support_x.max():.3f}")
print(f"   Query X - Min: {all_query_x.min():.3f}, Max: {all_query_x.max():.3f}")
print(f"   Mean: {all_support_x.mean():.3f}, Std: {all_support_x.std():.3f}")

# Tasks per segment
from collections import Counter
segment_counts = Counter([dataset[i]['segment_id'] for i in range(len(dataset))])
print(f"\n✅ Tasks per Segment:")
print(f"   Min: {min(segment_counts.values())}")
print(f"   Max: {max(segment_counts.values())}")
print(f"   Mean: {np.mean(list(segment_counts.values())):.1f}")

print(f"\n{'='*50}")
print(f"✅ VALIDATION COMPLETE - Dataset is ready for MAML training!")

## Step 12: Visualize Task Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Tasks per regime
axes[0, 0].bar(['Regime 0\n(Bearish)', 'Regime 1\n(Bullish)'], 
               [regime_0_tasks, regime_1_tasks],
               color=['#e74c3c', '#2ecc71'], alpha=0.7, edgecolor='black', linewidth=2)
axes[0, 0].set_ylabel('Task Count', fontsize=12)
axes[0, 0].set_title('Task Distribution Across Regimes', fontsize=14, fontweight='bold')
for i, count in enumerate([regime_0_tasks, regime_1_tasks]):
    axes[0, 0].text(i, count + 2, str(count), ha='center', va='bottom', 
                     fontsize=14, fontweight='bold')
axes[0, 0].grid(True, axis='y', alpha=0.3)

# Plot 2: Tasks per segment
segment_ids = list(segment_counts.keys())
task_counts = list(segment_counts.values())
axes[0, 1].bar(range(len(segment_ids)), task_counts, alpha=0.7, 
               edgecolor='black', color='steelblue')
axes[0, 1].axhline(np.mean(task_counts), color='red', linestyle='--', 
                    linewidth=2, label=f'Mean: {np.mean(task_counts):.1f}')
axes[0, 1].set_xlabel('Segment ID', fontsize=12)
axes[0, 1].set_ylabel('Tasks per Segment', fontsize=12)
axes[0, 1].set_title('Task Distribution Across Segments', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, axis='y', alpha=0.3)

# Plot 3: Feature value distribution
all_features_flat = all_support_x.flatten().numpy()
axes[1, 0].hist(all_features_flat, bins=50, edgecolor='black', alpha=0.7, color='purple')
axes[1, 0].axvline(0, color='red', linestyle='--', linewidth=2, label='Zero')
axes[1, 0].set_xlabel('Feature Value', fontsize=12)
axes[1, 0].set_ylabel('Count', fontsize=12)
axes[1, 0].set_title('Distribution of Feature Values', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Support vs Query feature means
support_means = [dataset[i]['support_x'].mean().item() for i in range(min(100, len(dataset)))]
query_means = [dataset[i]['query_x'].mean().item() for i in range(min(100, len(dataset)))]
axes[1, 1].scatter(support_means, query_means, alpha=0.5, s=30)
axes[1, 1].plot([min(support_means), max(support_means)], 
                 [min(support_means), max(support_means)], 
                 'r--', linewidth=2, label='y=x')
axes[1, 1].set_xlabel('Support Set Mean', fontsize=12)
axes[1, 1].set_ylabel('Query Set Mean', fontsize=12)
axes[1, 1].set_title('Support vs Query Feature Means', fontsize=14, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/figures/ma_task_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Task distribution saved to docs/figures/ma_task_distribution.png")

## Step 13: Final Summary Report

In [ ]:
print("=" * 60)
print("MA CROSSOVER REGIME DETECTION - FINAL SUMMARY")
print("=" * 60)
print()
print(f"📊 REGIME DETECTION:")
print(f"   Method: Moving Average Crossover (MA-20 vs MA-50)")
print(f"   Total regime switches: {num_switches}")
print(f"   Regime 0 (Bearish) days: {regime_0_days} ({regime_0_days/len(df_clean)*100:.1f}%)")
print(f"   Regime 1 (Bullish) days: {regime_1_days} ({regime_1_days/len(df_clean)*100:.1f}%)")
print()
print(f"📦 SEGMENTATION:")
print(f"   Raw segments: {len(segments_raw)}")
print(f"   Filtered segments (≥{MIN_SEGMENT_LENGTH} days): {len(segments)}")
print(f"   Regime 0 segments: {len(regime_0_segs)}")
print(f"   Regime 1 segments: {len(regime_1_segs)}")
print(f"   Average segment length: {avg_length:.1f} days")
print()
print(f"🎯 MAML DATASET:")
print(f"   Total tasks created: {len(dataset)}")
print(f"   Tasks per segment (avg): {len(dataset) / len(segments):.2f}")
print(f"   Regime 0 tasks: {regime_0_tasks} ({regime_0_tasks/len(dataset)*100:.1f}%)")
print(f"   Regime 1 tasks: {regime_1_tasks} ({regime_1_tasks/len(dataset)*100:.1f}%)")
print()
print(f"📐 TASK STRUCTURE:")
print(f"   Support set size: 30 days × 11 features")
print(f"   Query set size: 10 days × 11 features")
print(f"   Window stride: 20 days (50% overlap)")
print(f"   NaN tasks: {nan_count} / {len(dataset)}")
print()
print(f"💾 OUTPUT FILES:")
print(f"   ✅ data/processed/regime_labels_ma.csv")
print(f"   ✅ data/processed/regime_segments_ma.pkl")
print(f"   ✅ docs/figures/ma_crossover_regimes.png")
print(f"   ✅ docs/figures/ma_segment_distribution.png")
print(f"   ✅ docs/figures/ma_task_distribution.png")
print()
print("=" * 60)
print("✅ MA CROSSOVER DATASET CREATION COMPLETE!")
print("=" * 60)
print()
print("Next Steps:")
print("  1. ✅ Dataset is ready for MAML training (Epic 1.4)")
print("  2. 📊 Compare with HMM-based dataset")
print("  3. 🚀 Train MAML model on both datasets")
print("  4. 📈 Evaluate which regime detection works better")